# Supplementary Figure S4: Reference Conditioning and Centroid Analysis

- Figure / panels: `FigS4a` to `FigS4f`, with `FigS4g` exported as an optional reserve panel
- Inputs: resolved metrics files from `artifacts/results/<dataset>/...`, Systema baseline outputs, payload `pkl`
- Outputs: `artifacts/paper_figures/main/FigS4_ReferenceConditioning/*`
- Run order: run after the Fig2 benchmark notebook; this notebook focuses on nearest vs random and reference baselines
- Dataset selector: edit `SELECTED_DATASET_KEYS` in the parameter cell below
- Role: Main text


In [1]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel, stratified_benchmark, systema_mechanism

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item, load_metrics_df, resolve_result
from scripts.systema._core import baselines_core as systema_core
from scripts.common.split_utils import split_by_dataset_policy
from trishift._utils import normalize_condition
importlib.reload(baseline_panel)
importlib.reload(stratified_benchmark)
importlib.reload(systema_mechanism)


<module 'scripts.trishift.analysis.systema_mechanism' from '/data/yilangliu/trishift/scripts/trishift/analysis/systema_mechanism.py'>

In [2]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit', 'subgroup_filter': None},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson', 'subgroup_filter': None},
    {'dataset_key': 'norman', 'dataset_label': 'Norman', 'subgroup_filter': None},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562', 'subgroup_filter': None},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1', 'subgroup_filter': None},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['adamson', 'norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

REFERENCE_MODELS = ['trishift_nearest', 'trishift_random']  # edit here
DISTANCE_MODELS = ['trishift_nearest', 'systema_nonctl_mean']  # edit here
GENERIC_SHIFT_MODELS = ['trishift_nearest', 'systema_nonctl_mean']
NORMAN_SUBGROUP_MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
RESULT_MODE = 'default'
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS4_ReferenceConditioning'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_CASE_OVERRIDE = {'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}
MODEL_LABELS = {
    'trishift_nearest': 'TriShift nearest',
    'trishift_random': 'TriShift random',
    'biolord': 'biolord',
    'gears': 'GEARS',
    'genepert': 'GenePert',
    'scgpt': 'scGPT',
    'systema_nonctl_mean': 'Perturbed mean',
    'systema_matching_mean': 'Matching mean',
    'Truth': 'Truth',
}
MODEL_COLORS = {
    'TriShift': '#9FD9D3',
    'biolord': '#F0806A',
    'GEARS': '#F2B56B',
    'GenePert': '#87A8D8',
    'scGPT': '#DDD3C8',
    'Perturbed mean': '#B9AEDC',
    'Matching mean': '#8D7BC8',
    'TriShift nearest': '#9FD9D3',
    'TriShift random': '#C9E8E4',
    'Truth': '#7F7F7F',
}
print('Selected datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Result mode:', RESULT_MODE)
print('Reference case override:', REFERENCE_CASE_OVERRIDE)
print('Norman subgroup models:', NORMAN_SUBGROUP_MODELS)


Selected datasets: ['adamson', 'norman']
Selected splits: [1, 2, 3, 4, 5]
Result mode: default
Reference case override: {'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}
Norman subgroup models: ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt']


In [3]:
def _read_csv_optional(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 5 else pd.DataFrame()


def _run_analysis_cli(module_name: str, *, dataset_key: str, models: list[str], subgroup_filter: str | None, out_dir: Path) -> None:
    cmd = [
        sys.executable,
        '-m',
        module_name,
        '--dataset',
        dataset_key,
        '--models',
        ','.join(models),
        '--split_ids',
        ','.join(map(str, SPLIT_IDS)),
        '--out_root',
        str(out_dir),
        '--result_mode',
        RESULT_MODE,
    ]
    if subgroup_filter:
        cmd.extend(['--subgroup_filter', subgroup_filter])
    if module_name.endswith('systema_mechanism'):
        cmd.extend(['--paths_path', str(repo_root / 'configs' / 'paths.yaml')])
    subprocess.run(cmd, cwd=repo_root, check=True, capture_output=True, text=True)


def safe_reference_benchmark(dataset_key: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'reference_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        _run_analysis_cli(
            'scripts.trishift.analysis.baseline_panel',
            dataset_key=dataset_key,
            models=REFERENCE_MODELS,
            subgroup_filter=subgroup_filter,
            out_dir=out_dir,
        )
        return {
            'status': 'ready',
            'result': {
                'out_dir': out_dir,
                'summary_df': _read_csv_optional(out_dir / 'baseline_panel_summary.csv'),
                'raw_df': _read_csv_optional(out_dir / 'baseline_panel_raw.csv'),
                'ranking_df': _read_csv_optional(out_dir / 'baseline_panel_ranking.csv'),
            },
        }
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def safe_mechanism_summary(dataset_key: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'mechanism_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        _run_analysis_cli(
            'scripts.trishift.analysis.systema_mechanism',
            dataset_key=dataset_key,
            models=GENERIC_SHIFT_MODELS,
            subgroup_filter=subgroup_filter,
            out_dir=out_dir,
        )
        return {
            'status': 'ready',
            'result': {
                'out_dir': out_dir,
                'residual_summary_df': _read_csv_optional(out_dir / 'residualized_systema_summary.csv'),
                'centroid_summary_df': _read_csv_optional(out_dir / 'centroid_accuracy_summary.csv'),
                'difficulty_bin_df': _read_csv_optional(out_dir / 'difficulty_bin_generic_shift_summary.csv'),
            },
        }
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc)}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def _shrink_bars(ax: plt.Axes, scale: float = 0.72) -> None:
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


def render_grouped_bar(df: pd.DataFrame, metric_col: str, metric_label: str, out_path: Path, order: list[str] | None = None, hue_order: list[str] | None = None) -> None:
    fig, ax = plt.subplots(figsize=(5.2, 3.9), dpi=240)
    if df.empty:
        ax.text(0.5, 0.5, f'No rows for {metric_col}', ha='center', va='center')
        ax.axis('off')
    else:
        work = df.copy()
        if order is not None and 'dataset_label' in work.columns:
            work['dataset_label'] = pd.Categorical(work['dataset_label'], categories=order, ordered=True)
            work = work.sort_values('dataset_label')
        dataset_order = [label for label in (order or sorted(work['dataset_label'].astype(str).unique())) if label in set(work['dataset_label'].astype(str))]
        label_order = [label for label in (hue_order or list(dict.fromkeys(work['label'].astype(str)))) if label in set(work['label'].astype(str))]
        palette = {label: MODEL_COLORS.get(label, '#CCCCCC') for label in label_order}
        vals = pd.to_numeric(work[metric_col], errors='coerce').dropna()
        if vals.empty:
            ax.text(0.5, 0.5, f'No numeric rows for {metric_col}', ha='center', va='center')
            ax.axis('off')
        else:
            ymin, ymax = float(vals.min()), float(vals.max())
            span = max(ymax - ymin, max(abs(ymax), abs(ymin), 1e-6))
            ax.set_ylim(min(0.0, ymin - 0.08 * span), ymax + 0.28 * span)
            group_width = 0.82
            for dataset_idx, dataset_label in enumerate(dataset_order):
                sub = work[work['dataset_label'].astype(str) == str(dataset_label)].copy()
                present_labels = [label for label in label_order if label in set(sub['label'].astype(str))]
                if not present_labels:
                    continue
                base_width = min(0.16, group_width / max(len(present_labels), 1))
                bar_width = base_width * 0.9
                offsets = [0.0] if len(present_labels) == 1 else np.linspace(-base_width * (len(present_labels) - 1) / 2, base_width * (len(present_labels) - 1) / 2, len(present_labels))
                for offset, label in zip(offsets, present_labels):
                    row = sub[sub['label'].astype(str) == str(label)]
                    if row.empty:
                        continue
                    value = float(pd.to_numeric(row[metric_col], errors='coerce').iloc[0])
                    ax.bar(dataset_idx + float(offset), value, width=bar_width, color=palette[label], edgecolor='black', linewidth=0.5)
            ax.set_xticks(np.arange(len(dataset_order)))
            ax.set_xticklabels(dataset_order)
            ax.set_xlabel('')
            ax.set_ylabel(metric_label)
            ax.set_title(metric_label, fontsize=10, pad=8)
            ax.tick_params(axis='x', rotation=32)
            _style_axis(ax)
            handles = [plt.Rectangle((0, 0), 1, 1, facecolor=palette[label], edgecolor='black', linewidth=0.5) for label in label_order]
            ax.legend(handles, label_order, title='', frameon=False, ncol=min(3, len(label_order)), loc='upper left', bbox_to_anchor=(0.02, 0.995), borderaxespad=0.0, handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
    fig.tight_layout(); fig.savefig(out_path, bbox_inches='tight'); plt.close(fig)


def _dense_2d(x):
    if hasattr(x, 'toarray'):
        arr = x.toarray()
    else:
        arr = np.asarray(x)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    return arr


_systema_case_ctx_cache = {}

def _systema_case_payload(dataset_key: str, split_id: int, condition: str, model_name: str):
    if model_name != 'systema_nonctl_mean':
        return None
    key = (dataset_key, int(split_id))
    if key not in _systema_case_ctx_cache:
        paths_cfg = systema_core.load_yaml(str(repo_root / 'configs' / 'paths.yaml'))
        h5ad_path = Path(paths_cfg['datasets'][dataset_key])
        if not h5ad_path.is_absolute():
            h5ad_path = repo_root / h5ad_path
        emb_key = systema_core.DATASET_CONFIG[dataset_key]['emb_key']
        emb_path = Path(paths_cfg['embeddings'][emb_key])
        if not emb_path.is_absolute():
            emb_path = repo_root / emb_path
        adata = systema_core.load_adata(str(h5ad_path))
        adata.uns = {}
        embd_df = systema_core.load_embedding_df(str(emb_path))
        embd_df = systema_core.apply_alias_mapping(embd_df, dataset_key)
        degs_cache_path = repo_root / 'artifacts' / 'cache' / 'degs' / f'{dataset_key}_degs.pkl'
        degs_cache = systema_core._load_degs_cache(degs_cache_path)
        if degs_cache:
            adata.uns.update(degs_cache)
        data = systema_core.TriShiftData(adata, embd_df)
        data.setup_embedding_index()
        data.build_or_load_degs()
        split_dict = split_by_dataset_policy(data, dataset_name=dataset_key, seed=int(split_id))
        train_adata = split_dict['train']
        train_ctrl_adata = systema_core._ctrl_pool_from_split(data, train_adata)
        _systema_case_ctx_cache[key] = {
            'data': data,
            'train_ctrl_adata': train_ctrl_adata,
            'mu_pert': systema_core._systema_nonctl_mean_mu(train_adata),
            'cond_series': data.adata_all.obs[data.label_key].astype(str).values,
            'var_names': np.asarray(data.adata_all.var_names).astype(str),
            'X_all': data.adata_all.X,
        }
    ctx = _systema_case_ctx_cache[key]
    cond = normalize_condition(str(condition))
    data = ctx['data']
    deg_idx = np.asarray(systema_core._get_deg_idx(data, cond), dtype=int)
    if deg_idx.size == 0:
        return None
    truth_mask = ctx['cond_series'] == cond
    if not np.any(truth_mask):
        return None
    return {
        'Pred': np.asarray(ctx['mu_pert'][deg_idx], dtype=np.float32).reshape(1, -1),
        'Ctrl': _dense_2d(ctx['train_ctrl_adata'][:, deg_idx].X),
        'Truth': _dense_2d(ctx['X_all'][truth_mask][:, deg_idx]),
        'DE_name': ctx['var_names'][deg_idx],
    }


def _build_reference_case(dataset_key: str, condition: str, split_id: int):
    condition_key = normalize_condition(condition)
    loaded = {}
    for model_name in ['trishift_nearest', 'systema_nonctl_mean']:
        try:
            if model_name == 'systema_nonctl_mean':
                item = _systema_case_payload(dataset_key, split_id, condition_key, model_name)
            else:
                _, payload = load_payload_item(dataset=dataset_key, model_name=model_name, split_id=split_id, condition=None, result_mode=RESULT_MODE)
                item = {normalize_condition(k): v for k, v in payload.items()}.get(condition_key)
        except Exception:
            item = None
        if item is not None:
            loaded[model_name] = item
    if 'trishift_nearest' not in loaded:
        return None, None
    base_item = loaded['trishift_nearest']
    truth = np.asarray(base_item['Truth'], dtype=float)
    ctrl = np.asarray(base_item['Ctrl'], dtype=float)
    genes = np.asarray(base_item.get('DE_name', [f'g{i}' for i in range(np.asarray(base_item['Pred']).shape[1])])).astype(str)
    truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
    order = np.argsort(-np.abs(truth_delta))[:12]
    rows = [pd.DataFrame({'Gene': genes[order], 'Expression': truth_delta[order], 'Group': 'Truth'})]
    for model_name in ['trishift_nearest', 'systema_nonctl_mean']:
        if model_name not in loaded:
            continue
        item = loaded[model_name]
        pred_delta = np.asarray(item['Pred'], dtype=float).mean(axis=0) - np.asarray(item['Ctrl'], dtype=float).mean(axis=0)
        rows.append(pd.DataFrame({'Gene': genes[order], 'Expression': pred_delta[order], 'Group': MODEL_LABELS[model_name]}))
    return {'dataset_key': dataset_key, 'condition': condition_key, 'split_id': int(split_id)}, pd.concat(rows, ignore_index=True)


def pick_reference_case():
    if REFERENCE_CASE_OVERRIDE in (None, {}):
        raise ValueError('REFERENCE_CASE_OVERRIDE must be set explicitly')
    case_meta, case_df = _build_reference_case(
        dataset_key=REFERENCE_CASE_OVERRIDE['dataset_key'],
        condition=REFERENCE_CASE_OVERRIDE['condition'],
        split_id=int(REFERENCE_CASE_OVERRIDE['split_id']),
    )
    if case_meta is None or case_df is None:
        raise ValueError(f"Reference case could not be resolved: {REFERENCE_CASE_OVERRIDE}")
    return case_meta, case_df


In [4]:
reference_runs, mechanism_runs = [], []
for spec in DATASET_SPECS:
    reference_runs.append({**spec, **safe_reference_benchmark(spec['dataset_key'], subgroup_filter=spec.get('subgroup_filter'))})
    mechanism_runs.append({**spec, **safe_mechanism_summary(spec['dataset_key'], subgroup_filter=spec.get('subgroup_filter'))})

reference_frames = []
for row in reference_runs:
    if row['status'] == 'ready':
        df = row['result']['summary_df'].copy()
        df['dataset_label'] = row['dataset_label']
        reference_frames.append(df)
reference_summary_df = pd.concat(reference_frames, ignore_index=True) if reference_frames else pd.DataFrame()
reference_summary_df.to_csv(OUT_ROOT / 'reference_summary_all.csv', index=False, encoding='utf-8-sig')

residual_frames, centroid_frames, bin_frames = [], [], []
for row in mechanism_runs:
    if row['status'] != 'ready':
        continue
    spec = row['dataset_label']
    residual_df = row['result']['residual_summary_df'].copy()
    centroid_df = row['result']['centroid_summary_df'].copy()
    bin_df = row['result']['difficulty_bin_df'].copy()
    if not residual_df.empty:
        residual_df['dataset_label'] = row['dataset_label']
        residual_frames.append(residual_df)
    if not centroid_df.empty:
        centroid_df['dataset_label'] = row['dataset_label']
        centroid_frames.append(centroid_df)
    if not bin_df.empty:
        bin_df['dataset_label'] = row['dataset_label']
        bin_frames.append(bin_df)
residual_summary_df = pd.concat(residual_frames, ignore_index=True) if residual_frames else pd.DataFrame()
centroid_summary_df = pd.concat(centroid_frames, ignore_index=True) if centroid_frames else pd.DataFrame()
difficulty_bin_df = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
residual_summary_df.to_csv(OUT_ROOT / 'residualized_systema_summary_all.csv', index=False, encoding='utf-8-sig')
centroid_summary_df.to_csv(OUT_ROOT / 'centroid_accuracy_summary_all.csv', index=False, encoding='utf-8-sig')
difficulty_bin_df.to_csv(OUT_ROOT / 'difficulty_bin_generic_shift_summary_all.csv', index=False, encoding='utf-8-sig')
display(reference_summary_df.head())
display(residual_summary_df.head())
display(centroid_summary_df.head())

norman_subgroup_summary_df = pd.DataFrame()
norman_subgroup_df = pd.DataFrame()
if 'norman' in SELECTED_DATASET_KEYS:
    subgroup_dir = OUT_ROOT / 'norman_subgroup_all'
    subgroup_dir.mkdir(parents=True, exist_ok=True)
    try:
        _run_analysis_cli(
            'scripts.trishift.analysis.baseline_panel',
            dataset_key='norman',
            models=NORMAN_SUBGROUP_MODELS,
            subgroup_filter=None,
            out_dir=subgroup_dir,
        )
        norman_subgroup_summary_df = _read_csv_optional(subgroup_dir / 'baseline_panel_summary.csv')
        norman_subgroup_df = _read_csv_optional(subgroup_dir / 'norman_subgroup_panel.csv')
        if not norman_subgroup_summary_df.empty:
            norman_subgroup_summary_df['label'] = norman_subgroup_summary_df['model_name'].map(MODEL_LABELS).fillna(norman_subgroup_summary_df.get('label', norman_subgroup_summary_df['model_name']))
        if not norman_subgroup_df.empty:
            norman_subgroup_df['label'] = norman_subgroup_df['model_name'].map(MODEL_LABELS).fillna(norman_subgroup_df.get('label', norman_subgroup_df['model_name']))
    except Exception as exc:
        print(f'Norman subgroup benchmark pending: {exc}')
norman_subgroup_summary_df.to_csv(OUT_ROOT / 'figs4_norman_subgroup_summary.csv', index=False, encoding='utf-8-sig')
norman_subgroup_df.to_csv(OUT_ROOT / 'figs4_norman_subgroup_metrics.csv', index=False, encoding='utf-8-sig')
display(norman_subgroup_df.head())


,dataset,model_name,label,n_rows,split_ids_used,metrics_path,mean_pearson,mean_nmse,mean_deg_mean_r2,mean_systema_corr_20de_allpert,mean_systema_corr_deg_r2,mean_scpram_r2_degs_mean_mean,mean_scpram_r2_degs_var_mean,mean_scpram_wasserstein_degs_sum,dataset_label
0,adamson,trishift_nearest,TriShift nearest,85,"1,2,3,4,5",/data/yilangliu/trishift/artifacts/results/ada...,0.942082,0.172160,0.800025,0.535673,-0.176765,0.949250,0.842562,4.027908,Adamson
1,adamson,trishift_random,TriShift random,85,"1,2,3,4,5",/data/yilangliu/trishift/artifacts/results/ada...,0.925300,0.194610,0.777359,0.449672,-0.156254,0.941086,0.785665,4.168254,Adamson
2,norman,trishift_nearest,TriShift nearest,230,"1,2,3,4,5",/data/yilangliu/trishift/artifacts/results/nor...,0.869904,0.240757,0.687961,0.783209,0.526281,0.961278,0.512200,5.663613,Norman
3,norman,trishift_random,TriShift random,230,"1,2,3,4,5",/data/yilangliu/trishift/artifacts/results/nor...,0.872867,0.263702,0.662995,0.746507,0.475514,0.950916,0.490639,5.964471,Norman


,dataset,model_name,systema_corr_20de_allpert,residualized_systema_corr_20de_allpert,generic_projection_ratio,dataset_label
0,adamson,systema_nonctl_mean,0.339018,0.212542,0.999306,Adamson
1,adamson,trishift_nearest,0.533082,0.462631,0.828281,Adamson
2,norman,systema_nonctl_mean,-0.288134,0.146717,0.995030,Norman
3,norman,trishift_nearest,0.784280,0.790175,0.503676,Norman


,dataset,model_name,centroid_accuracy,centroid_top1,centroid_top3,centroid_mean_rank,n_conditions,dataset_label
0,adamson,systema_nonctl_mean,0.500000,0.058824,0.176471,9.000000,17.0,Adamson
1,adamson,trishift_nearest,0.650000,0.129412,0.352941,6.600000,17.0,Adamson
2,norman,systema_nonctl_mean,0.500000,0.021743,0.065230,23.500000,46.0,Norman
3,norman,trishift_nearest,0.935642,0.660711,0.790990,3.886885,46.0,Norman


,dataset,model_name,label,subgroup,n_rows,mean_pearson,mean_nmse,mean_deg_mean_r2,mean_systema_corr_20de_allpert,mean_systema_corr_deg_r2,mean_scpram_r2_degs_mean_mean,mean_scpram_r2_degs_var_mean,mean_scpram_wasserstein_degs_sum
0,norman,trishift_nearest,TriShift nearest,seen0,6,0.949524,0.107931,0.874621,0.924216,0.826534,0.951382,0.362237,6.115620
1,norman,trishift_nearest,TriShift nearest,seen1,48,0.961550,0.109735,0.859260,0.921648,0.765100,0.963796,0.444073,5.845212
2,norman,trishift_nearest,TriShift nearest,seen2,71,0.972913,0.097422,0.885660,0.943103,0.798992,0.969745,0.514658,5.690464
3,norman,trishift_nearest,TriShift nearest,single,105,0.753807,0.405164,0.465305,0.603746,0.215545,0.954968,0.550250,5.536610
4,norman,biolord,biolord,seen0,6,0.930809,0.204790,0.709580,0.818478,0.517220,NaN,NaN,NaN


In [5]:
dataset_order = [spec['dataset_label'] for spec in DATASET_SPECS]
nearest_random_df = reference_summary_df[reference_summary_df['model_name'].isin(['trishift_nearest', 'trishift_random'])].copy()
if not nearest_random_df.empty:
    nearest_random_df['label'] = nearest_random_df['model_name'].map(MODEL_LABELS).fillna(nearest_random_df['model_name'])
render_grouped_bar(nearest_random_df, 'mean_pearson', 'Reference retrieval', OUT_ROOT / 'figs4a_nearest_vs_random.png', order=dataset_order, hue_order=['TriShift nearest', 'TriShift random'])

systema_plot_df = residual_summary_df[residual_summary_df.get('model_name', pd.Series(dtype=str)).isin(GENERIC_SHIFT_MODELS)].copy() if 'model_name' in residual_summary_df.columns else pd.DataFrame()
if not systema_plot_df.empty:
    systema_plot_df['label'] = systema_plot_df['model_name'].map(MODEL_LABELS).fillna(systema_plot_df['model_name'])
render_grouped_bar(systema_plot_df, 'systema_corr_20de_allpert', 'Systema Pearson', OUT_ROOT / 'figs4b_systema_pearson.png', order=dataset_order, hue_order=['TriShift nearest', 'Perturbed mean'])

signal_df = reference_summary_df[reference_summary_df['model_name'].isin(REFERENCE_MODELS)].copy()
metric_cols = ['mean_pearson', 'mean_nmse']
if not signal_df.empty:
    signal_df['label'] = signal_df['model_name'].map(MODEL_LABELS).fillna(signal_df['model_name'])
    ref_heatmap = signal_df[['label', 'dataset_label'] + [c for c in metric_cols if c in signal_df.columns]].copy().melt(id_vars=['label', 'dataset_label'], var_name='metric', value_name='value')
    ref_heatmap['metric'] = ref_heatmap['metric'].str.replace('mean_', '', regex=False)
    ref_heatmap['column'] = ref_heatmap['dataset_label'] + '\n' + ref_heatmap['metric']
    ref_table = ref_heatmap.pivot_table(index='label', columns='column', values='value', aggfunc='mean')
else:
    ref_table = pd.DataFrame()
fig, ax = plt.subplots(figsize=(max(9, ref_table.shape[1] * 1.0 if not ref_table.empty else 9), 4.8), dpi=220)
if ref_table.empty:
    ax.text(0.5, 0.5, 'No reference-baseline summary available', ha='center', va='center'); ax.axis('off')
else:
    sns.heatmap(ref_table, annot=False, cmap='viridis', linewidths=0.0, linecolor=None, ax=ax, cbar_kws={'shrink': 0.88, 'pad': 0.02})
    ax.set_title('TriShift vs reference baselines')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelrotation=58, labelsize=8)
    ax.tick_params(axis='y', labelrotation=0, labelsize=9)
    for tick in ax.get_xticklabels():
        tick.set_ha('right')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)
plt.tight_layout(); plt.savefig(OUT_ROOT / 'figs4c_reference_baselines.png', bbox_inches='tight'); plt.close()

figs4d_hue_order = ['TriShift nearest', 'Perturbed mean']
residual_plot_df = residual_summary_df[residual_summary_df.get('model_name', pd.Series(dtype=str)).isin(GENERIC_SHIFT_MODELS)].copy() if 'model_name' in residual_summary_df.columns else pd.DataFrame()
if not residual_plot_df.empty:
    residual_plot_df['label'] = residual_plot_df['model_name'].map(MODEL_LABELS).fillna(residual_plot_df['model_name'])
render_grouped_bar(
    residual_plot_df,
    'residualized_systema_corr_20de_allpert',
    'Residualized Systema Pearson',
    OUT_ROOT / 'figs4d_residualized_systema_pearson.png',
    order=dataset_order,
    hue_order=figs4d_hue_order,
)

figs4e_hue_order = ['TriShift nearest', 'Perturbed mean']
centroid_plot_df = centroid_summary_df[centroid_summary_df.get('model_name', pd.Series(dtype=str)).isin(GENERIC_SHIFT_MODELS)].copy() if 'model_name' in centroid_summary_df.columns else pd.DataFrame()
if not centroid_plot_df.empty:
    centroid_plot_df['label'] = centroid_plot_df['model_name'].map(MODEL_LABELS).fillna(centroid_plot_df['model_name'])
render_grouped_bar(
    centroid_plot_df,
    'centroid_accuracy',
    'Centroid accuracy',
    OUT_ROOT / 'figs4e_centroid_accuracy.png',
    order=dataset_order,
    hue_order=figs4e_hue_order,
)


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.9), dpi=220)
plot_specs = [
    ('systema_corr_20de_allpert', 'Systema Pearson'),
    ('residualized_systema_corr_20de_allpert', 'Residualized Systema Pearson'),
    ('generic_projection_ratio', 'Generic-shift projection ratio'),
]
required_cols = {'model_name', 'dataset_label', 'train_distance_bin'}
if difficulty_bin_df.empty or not required_cols.issubset(difficulty_bin_df.columns):
    for ax in axes:
        ax.text(0.5, 0.5, 'No difficulty-bin summary available', ha='center', va='center')
        ax.axis('off')
else:
    from matplotlib.lines import Line2D

    work = difficulty_bin_df[difficulty_bin_df['model_name'].isin(GENERIC_SHIFT_MODELS)].copy()
    work['label'] = work['model_name'].map(MODEL_LABELS).fillna(work['model_name'])
    work['train_distance_bin'] = pd.Categorical(work['train_distance_bin'], categories=['near', 'medium', 'far'], ordered=True)
    dataset_style = {'Adamson': ('o', '-'), 'Norman': ('X', '--')}
    ordered_labels = [MODEL_LABELS[m] for m in GENERIC_SHIFT_MODELS if m in set(work['model_name'])]
    for ax, (metric_col, title) in zip(axes, plot_specs):
        if metric_col not in work.columns:
            ax.text(0.5, 0.5, f'Missing {metric_col}', ha='center', va='center')
            ax.axis('off')
            continue
        for label in ordered_labels:
            for dataset_label in dataset_order:
                sub = work[(work['label'] == label) & (work['dataset_label'] == dataset_label)].copy()
                if sub.empty:
                    continue
                marker, linestyle = dataset_style.get(dataset_label, ('o', '-'))
                sns.lineplot(
                    data=sub,
                    x='train_distance_bin',
                    y=metric_col,
                    sort=False,
                    marker=marker,
                    linestyle=linestyle,
                    color=MODEL_COLORS[label],
                    linewidth=2.0,
                    markersize=7,
                    legend=False,
                    ax=ax,
                )
        ax.set_xlabel('Train-distance bin')
        ax.set_ylabel(title)
        ax.set_title(title)
        _style_axis(ax)
    model_handles = [Line2D([0], [0], color=MODEL_COLORS[label], lw=2.2, label=label) for label in ordered_labels]
    dataset_handles = [
        Line2D([0], [0], color='#444444', lw=1.8, marker='o', linestyle='-', label='Adamson'),
        Line2D([0], [0], color='#444444', lw=1.8, marker='X', linestyle='--', label='Norman'),
    ]
    leg1 = axes[0].legend(handles=model_handles, title='', frameon=False, loc='upper left', bbox_to_anchor=(0.0, 1.22), ncol=min(3, len(model_handles)), borderaxespad=0.0)
    axes[0].add_artist(leg1)
    axes[-1].legend(handles=dataset_handles, title='', frameon=False, loc='upper right', bbox_to_anchor=(1.0, 1.22), ncol=2, borderaxespad=0.0)
    fig.text(0.5, 0.01, 'Note: In the single-perturbation setting, Matching mean is omitted because it is identical to Perturbed mean.', ha='center', va='bottom', fontsize=9, color='#555555')
fig.tight_layout(rect=[0.0, 0.04, 1.0, 0.89]); plt.savefig(OUT_ROOT / 'figs4g_generic_shift_dependence.png', bbox_inches='tight'); plt.close()
display(difficulty_bin_df.head())



,dataset,model_name,train_distance_bin,systema_corr_20de_allpert,residualized_systema_corr_20de_allpert,generic_projection_ratio,dataset_label
0,adamson,systema_nonctl_mean,far,0.100528,-0.022807,0.999278,Adamson
1,adamson,systema_nonctl_mean,medium,0.496597,0.367804,0.999321,Adamson
2,adamson,systema_nonctl_mean,near,0.428446,0.301035,0.999319,Adamson
3,adamson,trishift_nearest,far,0.332320,0.323621,0.805623,Adamson
4,adamson,trishift_nearest,medium,0.529019,0.410950,0.847671,Adamson


In [8]:
case_meta, case_df = pick_reference_case()
if case_meta is not None and case_df is not None:
    case_df.to_csv(OUT_ROOT / 'figs4f_reference_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(11.0, 5.8), dpi=220)
    ax = sns.barplot(data=case_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    _shrink_bars(ax, scale=0.76)
    for patch in ax.patches:
        patch.set_edgecolor('black'); patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(f"Reference-sensitive case: {case_meta['condition']} (split {case_meta['split_id']})")
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=min(3, case_df['Group'].nunique()))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8)
    _style_axis(ax)
    plt.tight_layout(); plt.savefig(OUT_ROOT / 'figs4f_reference_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_df.head(20))
print(OUT_ROOT)



[embd] embedding index ready: key=embd_index
[degs] computing with scanpy


/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: 

/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:440: Perf

/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:440: Perf

/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:450: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "pvals_adj"] = pvals_adj[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:461: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "logfoldchanges"] = np.log2(
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:435: Performance

/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:440: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "pvals"] = pvals[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:450: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "pvals_adj"] = pvals_adj[global_indices]
/data/yilangliu/envs/trishift-cu128/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:461: Perform

[degs] scanpy degs ready
[degs] using precomputed degs: key=top20_degs_non_dropout


{'dataset_key': 'norman', 'condition': 'CITED1+ctrl', 'split_id': 4}


,Gene,Expression,Group
0,NEAT1,0.949279,Truth
1,MALAT1,0.874098,Truth
2,NME1,-0.508551,Truth
3,PTMA,-0.498049,Truth
4,RANBP1,-0.488963,Truth
5,CSF3R,0.484617,Truth
6,PHF19,-0.481764,Truth
7,CKS1B,-0.472855,Truth
8,LDHA,-0.462323,Truth
9,SRM,-0.458838,Truth


/data/yilangliu/trishift/artifacts/paper_figures/main/Fig3_ReferenceConditioning
